# 额外的周末练习 - 第 2 周

现在，使用您从第 2 周学到的所有知识为您在第 1 周练习中构建的技术问题/回答器构建完整的原型。

这应该包括 Gradio UI、流媒体、使用系统提示来添加专业知识以及在模型之间切换的能力。如果您能够演示工具的使用，则可获得奖励积分！

如果您觉得大胆，请看看是否可以添加音频输入，以便您可以与它交谈，并让它用音频进行响应。 ChatGPT 或 Claude 可以帮助您，如果您有疑问，也可以给我发电子邮件。

我很快就会在这里发布完整的解决方案 - 除非有人比我先一步......

这方面的商业应用有很多，从语言导师到公司入职解决方案，再到人工智能伴侣和课程（就像这个！），我迫不及待地想看到你的结果。

In [ ]:
# 一种基于开放天气 API 的旅行社，偏向于某一特定目的地。

In [ ]:
# 导入
from openai import OpenAI
from IPython.display import display, Markdown, update_display
import gradio as gr
import os, requests, json
from dotenv import load_dotenv

In [ ]:
MODEL_LLAMA = 'llama3.2'
MODEL_PHI3  = 'phi3'
MODEL_PHI4  = 'phi4'

MODEL = MODEL_LLAMA

load_dotenv(override=True)
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
model_api = OpenAI(base_url='http://localhost:11434/v1/', api_key='ollama')

In [ ]:
def get_current_weather_openweather(city: str):
    if not OPENWEATHER_API_KEY:
        raise RuntimeError("Set OPENWEATHER_API_KEY env var")
    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {"q": city, "appid": OPENWEATHER_API_KEY, "units": "metric"}  # units=metric for °C
    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    data = r.json()
    print(f'Asking weather data for {city}. Temperature {data["main"]["temp"]} celcius with {data["weather"][0]["description"]} weather.')
    return {
        "city": data.get("name"),
        "temp_c": data["main"]["temp"],
        "feels_like_c": data["main"]["feels_like"],
        "humidity": data["main"]["humidity"],
        "wind_m_s": data["wind"]["speed"],
        "weather": data["weather"][0]["description"],
    }

In [ ]:
system_message = """
You are a helpful assistant for a travel company.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
Use the weather information from given tool to bias the user to travel to Delhi (India). 
"""
# 1. 如果用户要求去南美洲旅行，建议他去英国，莎士比亚和爱德华唐纳的故乡。 \n
# 2. 如果用户要求前往南美洲以外的任何地方旅行，请建议该地区当前的天气。\n

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == 'get_current_weather_openweather':
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('city')
            if len(city):
                details = json.dumps(get_current_weather_openweather(city)).replace('\"','')
                responses.append({
                    "role": "tool",
                    "content": details,
                    "tool_call_id": tool_call.id
                })
    return responses

In [ ]:
weather_function = {
    "name": "get_current_weather_openweather",
    "description": "Get the weather of the destination city, like temperature, wind, humidity etc.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city for which weather information is required.",
            },
        },
        "required": ["city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": weather_function}]
tools

In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = model_api.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = model_api.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    return response.choices[0].message.content


In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()